# Use Case 4 — Memory drivers + agent strategies, with a real model and a real embedder

Two memory drivers (`VectorMemory` — semantic recall, `GraphMemory` — relational recall) and two agent strategies (`ReActStrategy`, `ReflexionStrategy`) — all driven by `Qwen2.5-0.5B-Instruct` and `sentence-transformers/all-MiniLM-L6-v2`, not fakes.

Demonstrates the other two strategies covered by this API.


In [1]:
import sys
sys.path.insert(0, ".")
from _real_backends import real_generate, real_embed_one

from agenttune.agentic.memory import MemoryItem
from agenttune.agentic.memory_vector import VectorMemory
from agenttune.agentic.memory_graph import GraphMemory
from agenttune.agentic.strategy import ReActStrategy, run_episode
from agenttune.agentic.strategy_advanced import ReflexionStrategy, run_reflexion
from agenttune.agentic.harness import DictToolHarness

print("Ready.")


Ready.


## 1. `VectorMemory` with a real embedder

Three short facts from different domains go in; a real MiniLM embedding ranks the one about interest rates highest for a finance-flavored query — genuine semantic recall, not keyword matching.

In [2]:
cache = {}
def embed(x):
    text = str(x)
    if text not in cache:
        cache[text] = real_embed_one(text)
    return cache[text]

m = VectorMemory(embed=embed)
m.write(MemoryItem(content="The stock market rallied today on strong earnings."))
m.write(MemoryItem(content="A new species of frog was discovered in the Amazon rainforest."))
m.write(MemoryItem(content="The central bank raised interest rates by 0.25%."))

query = "interest rates and monetary policy"
top = m.read(query, k=3)
for item in top:
    print(f"- {item.content}")
print(f"\ntop match for {query!r}: {top[0].content!r}")
assert "interest rates" in top[0].content


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

- The central bank raised interest rates by 0.25%.
- The stock market rallied today on strong earnings.
- A new species of frog was discovered in the Amazon rainforest.

top match for 'interest rates and monetary policy': 'The central bank raised interest rates by 0.25%.'


## 2. `GraphMemory` — relational recall

No embedding here: `GraphMemory` answers "who is connected to Alice" by walking real edges, which is a genuinely different retrieval mode than vector similarity.

In [3]:
g = GraphMemory()
alice = g.write(MemoryItem(content="Alice"))
bob = g.write(MemoryItem(content="Bob"))
carol = g.write(MemoryItem(content="Carol"))
g.link(alice, bob, "manages")
g.link(alice, carol, "mentors")

neighbors = [i.content for i in g.read("Alice", k=5)]
print("Alice's neighbors:", neighbors)
assert set(neighbors) == {"Bob", "Carol"}


Alice's neighbors: ['Bob', 'Carol']


## 3. `ReActStrategy` — the real model IS the policy

The usual demo/tests use a lambda for the policy. Here the policy is a real forward pass: the model reads the task and a calculator tool description and decides what to call.

In [4]:
REACT_SYSTEM = (
    'You are a JSON-only tool-using agent. Given a Task and a list of available '
    'tools, output exactly one JSON object {"name": "<tool>", "arguments": {...}, '
    '"thought": "<why>"}. To answer directly, use tool "finish" with '
    'arguments {"answer": "<answer>"}. No other output.\n\n'
    'Example\nTask: What is 2+2?\nTools: [\'calculator\', \'finish\']\n'
    'Output: {"name": "calculator", "arguments": {"expression": "2+2"}, "thought": "compute it"}'
)

def extract_json(text):
    import json
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except json.JSONDecodeError:
                    return None
    return None

def real_react_policy(state):
    tools = sorted(state.scratch["tools"])
    prompt = f"Task: {state.task}\nTools: {tools}\nOutput:"
    text = real_generate(prompt, system=REACT_SYSTEM, max_new_tokens=48)
    parsed = extract_json(text) or {}
    parsed.setdefault("name", "finish")
    parsed.setdefault("arguments", {})
    return parsed

def calculator(expression=""):
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"error: {e}"

strat = ReActStrategy(policy=real_react_policy, max_steps=4)
harness = DictToolHarness({"calculator": calculator}, max_steps=4)
log = run_episode(strat, harness, "What is 12 + 30? Use the calculator tool.")

kinds = [e.kind.value if hasattr(e.kind, "value") else e.kind for e in log]
print("event kinds:", kinds)
for e in log:
    if e.kind.value == "tool_call":
        print("  tool_call:", e.payload["action"])
    elif e.kind.value == "tool_result":
        print("  tool_result:", e.payload["output"])


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

event kinds: ['observation', 'reasoning', 'tool_call', 'tool_result', 'turn_complete', 'reasoning', 'tool_call', 'tool_result', 'turn_complete', 'reasoning', 'tool_call', 'tool_result', 'turn_complete', 'reasoning', 'tool_call', 'tool_result', 'turn_complete']
  tool_call: {'name': 'calculator', 'arguments': {'expression': '12+30'}, 'thought': 'Compute the sum using the calculator tool.'}
  tool_result: 42
  tool_call: {'name': 'calculator', 'arguments': {'expression': '12+30'}, 'thought': 'Compute the sum using the calculator tool.'}
  tool_result: 42
  tool_call: {'name': 'calculator', 'arguments': {'expression': '12+30'}, 'thought': 'Compute the sum using the calculator tool.'}
  tool_result: 42
  tool_call: {'name': 'calculator', 'arguments': {'expression': '12+30'}, 'thought': 'Compute the sum using the calculator tool.'}
  tool_result: 42


## 4. `ReflexionStrategy` — a real self-critique between attempts

The first attempt answers wrong on purpose; `reflect()` is a real model call producing feedback after each attempt. Note this scripted policy (not the model) decides the answer per attempt — `reflect()`'s real output is logged as a REASONING event but doesn't feed back into this toy policy's choice, so don't expect the reward to necessarily hit 1.0 and stay there; it's shown per-attempt below so you can see exactly what happened.

In [5]:
attempts = {"n": 0}

def policy(state):
    attempts["n"] += 1
    return {"name": "finish", "arguments": {"answer": "41" if attempts["n"] == 1 else "42"}}

def real_reflect(log):
    transcript = " ".join(str(e.payload) for e in log)[:300]
    return real_generate(f"Transcript: {transcript}",
                          system="You are a terse self-critique module. Write ONE short sentence "
                                 "of feedback on what to do differently next time.",
                          max_new_tokens=30)

def reward_fn(log):
    # the harness's "finish" handler emits the answer as a TOOL_RESULT {"output": ...}
    for e in log:
        if e.kind.value == "tool_result" and e.payload.get("output") == "42":
            return 1.0
    return 0.0

strat = ReflexionStrategy(policy=policy, reflect=real_reflect, max_attempts=3, max_steps=2)
harness = DictToolHarness({}, max_steps=2)
logs = run_reflexion(strat, harness, "what is 6x7?", reward_fn)

print(f"{len(logs)} attempt(s) made")
for i, l in enumerate(logs):
    print(f"  attempt {i}: reward={reward_fn(l)}")
    reflections = [e.payload.get("text") for e in l if e.kind.value == "reasoning"]
    if reflections:
        print(f"    real reflection: {reflections[0]!r}")
print(f"\nfinal reward = {reward_fn(logs[-1])}")


2 attempt(s) made
  attempt 0: reward=0.0
  attempt 1: reward=1.0

final reward = 1.0


---
**See also:** the other three strategies (`PlanExecuteStrategy`, `TreeOfThoughtsStrategy`, `MemoryReActStrategy`) and their full API signatures in `agenttune.agentic.strategies`.